# 第 57 课｜奖励信号、偏好数据与后训练目标

**从“挑出一个回答”，走向“改变以后生成回答的分布”。**

上一课的答案投票与 Best-of-N 不更新生成参数。本课分别训练一个评分模型和一个三动作生成分布，核对究竟谁收到了梯度。

| 本课目标 | 需要完成的检查 |
|---|---|
| 区分评分、偏好与训练目标 | 不把分数自动当成正确概率 |
| 构造可审计的偏好对 | 同题比较、先划分来源、跳过未知结果与平分 |
| 理解奖励模型的排序损失 | 核对稳定公式、梯度方向与平移不变性 |
| 区分 SFT、DPO 和奖励优化 | 说明各自更新谁、需要什么信号 |
| 检查混杂与代理目标 | 训练排序全对不代表新条件也能排序正确 |

**范围：CPU、PyTorch、完全离线。** 不下载模型、语料或框架，不运行真实 Qwen，不实现完整 PPO/GRPO。

本课训练对象只有两类：**2个参数的线性奖励模型**和**3个logits的有限动作策略**。下面所有数值都是教学实验，不是语言、编程或推理能力成绩。

## 0. 运行方式

沿用已有 PyTorch 的 Jupyter 内核，从上到下按 `Shift + Enter`。较长“实现”单元可折叠，但仍要执行；短单元展示需要观察的现象。

本 Notebook 已嵌入全部实现，**不依赖同名 `.py` 或前几课文件**。脚本也可以独立运行：

```bash
python lesson_57_rewards_preferences_and_posttraining.py --checks-only
python lesson_57_rewards_preferences_and_posttraining.py
```

先完成数据与奖励模型，再做目标函数与练习，适合分两次学习。练习检查默认关闭；运行全部单元成功不等于完成练习。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第57课：奖励信号、偏好数据与后训练目标。

Python 3.10+ 与 PyTorch；CPU、离线、无模型下载。
训练对象只有二维线性奖励模型和三动作的可训练概率表，不是语言模型成绩。
所有语料和实验均为公开教学数据；保存的状态用于重载，不用于精确断点续训。
"""
from __future__ import annotations


import argparse
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import re
import sys
import tempfile
from collections import Counter
from datetime import datetime, timezone
from fractions import Fraction
from pathlib import Path

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要现有 PyTorch 环境。请使用与 Notebook 相同的 Python 解释器。") from exc

VERSION = "lesson57-v1"
GRADER_VERSION = "exact_fraction_only-v1"
DTYPE = torch.float64
NUMBER_RE = re.compile(r"[+-]?[0-9]+(?:/[0-9]+|\.[0-9]+)?\Z")
SOURCES = {
    "RLHF": "https://arxiv.org/abs/2203.02155",
    "reward_model": "https://huggingface.co/docs/trl/reward_trainer",
    "DPO_paper": "https://arxiv.org/abs/2305.18290",
    "DPO_docs": "https://huggingface.co/docs/trl/dpo_trainer",
    "overoptimization": "https://arxiv.org/abs/2210.10760",
    "GRPO": "https://arxiv.org/abs/2402.03300",
    "log_softmax": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html",
}


class LessonError(ValueError):
    """输入或教学契约不满足要求。"""


def require(condition: bool, message: str) -> None:
    if not condition:
        raise LessonError(message)


def canonical(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def tensor_digest(tensor: torch.Tensor) -> str:
    t = tensor.detach().cpu().contiguous()
    raw = bytes(t.reshape(-1).view(torch.uint8).tolist())
    return hashlib.sha256(str((tuple(t.shape), str(t.dtype))).encode() + raw).hexdigest()


def strict_json(text: str):
    require(type(text) is str and len(text.encode("utf-8")) <= 4096,
            "JSON 必须是最多4096字节的字符串")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "重复 JSON 键")
            result[key] = value
        return result
    def invalid(_):
        raise LessonError("不接受NaN或Infinity")
    try:
        return json.loads(text, object_pairs_hook=pairs, parse_constant=invalid)
    except (json.JSONDecodeError, RecursionError) as exc:
        raise LessonError("JSON 格式错误") from exc


def parse_number(value: str) -> Fraction:
    require(type(value) is str and 0 < len(value) <= 64, "数值必须是有界字符串")
    value = value.strip()
    require(bool(NUMBER_RE.fullmatch(value)), "只接受一个整数、小数或分数")
    try:
        return Fraction(value)
    except (ValueError, ZeroDivisionError) as exc:
        raise LessonError("非法有理数") from exc


def write_json(path: Path, data) -> None:
    path.write_text(json.dumps(data, ensure_ascii=False, sort_keys=True,
                               indent=2, allow_nan=False) + "\n", encoding="utf-8")


def write_jsonl(path: Path, records) -> None:
    path.write_text("".join(canonical(r) + "\n" for r in records), encoding="utf-8")

In [2]:
torch.set_num_threads(1)
torch.manual_seed(57)
OUTPUT_ROOT = Path("lesson_57_outputs")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("本课实际设备：CPU；精度：float64")
print("当前工作目录：", Path.cwd())

Python: 3.13.5
PyTorch: 2.10.0+cpu
本课实际设备：CPU；精度：float64
当前工作目录： /mnt/data/_lesson57_work/notebook_run


## 1. 一张地图：同一个分数可以有不同用途

| 做法 | 使用的信号 | 改变什么 |
|---|---|---|
| Best-of-N | 候选评分 | 选择当前返回的候选，不更新生成参数 |
| SFT | 示范回答 | 更新生成模型，提高示范序列的对数似然 |
| 奖励模型 RM | 同题偏好对 | 更新评分模型，使较好回答获得较高分 |
| DPO | 偏好对＋固定参考策略 | 直接更新生成策略，不另训显式奖励模型 |
| 奖励驱动策略训练 | 采样回答与奖励 | 通过策略目标更新生成模型 |

经典 RLHF 流程包含示范微调、奖励建模与策略优化；它们不是同一次训练，也不是把评分器输出直接当作 token 标签。[InstructGPT 论文](https://arxiv.org/abs/2203.02155)

**LoRA / QLoRA 是另一层概念。** 它们决定哪些参数可训练及基座如何存储；上表决定训练信号与目标。使用 LoRA 并不能说明目标是 SFT、DPO 还是 RL。

本课的数据构造、人工特征RM、三动作策略是三个独立的最小实验。不要把它们误读为已经跑通真实语言模型的后训练流水线。

## 2. 偏好对：同一个问题，两份已经取得的回答

常见格式：

```json
{
  "prompt": "计算2+5，只输出约定JSON。",
  "chosen": "{\"answer\":\"7\"}",
  "rejected": "{\"answer\":\"8\"}"
}
```

两个 completion 必须对应同一提示。`chosen` 表示在这一对和评价标准下更受偏好，**一般情况下不保证它绝对正确，更不表示 rejected 每个字都错**。[TRL 数据约定](https://huggingface.co/docs/trl/dpo_trainer)

本课采取保守规则：只比较“完整生成、格式合法”的数值答案。精确正确得1，明确错误得0；同分不强行排序。缺失、生成错误和截断保持未知，不伪造成坏答案。

格式错误在本课也排除。其他项目可以把格式作为奖励，但要预先声明，不能把基础设施故障与回答质量混在一起。

In [3]:
def grade_candidate(task: dict, record: dict | None) -> dict:
    """未知/运行失败/截断不伪装成低质量答案；本课还排除格式错误。"""
    if record is None:
        return {"state": "missing", "score": None, "eligible": False}
    require(record.get("task_id") == task["task_id"], "回答与题目不匹配")
    require(record.get("status") in {"ok", "error"}, "未知生成状态")
    if record["status"] == "error":
        return {"state": "generation_error", "score": None, "eligible": False}
    if record.get("stop_reason") != "eos":
        return {"state": "incomplete", "score": None, "eligible": False}
    try:
        answer = strict_json(record.get("output"))
        require(type(answer) is dict and set(answer) == {"answer"}, "答案字段不匹配")
        value = parse_number(answer["answer"])
    except (LessonError, UnicodeError):
        return {"state": "invalid_format", "score": None, "eligible": False}
    correct = value == parse_number(task["expected"])
    return {"state": "graded", "score": float(correct), "eligible": True,
            "normalized_answer": str(value), "format_valid": True}


def make_fixture() -> tuple[list[dict], list[dict], dict]:
    """12来源，每来源两种提示；先按来源固定8/2/2切分，之后才生成偏好。"""
    groups = [f"source-{i:02}" for i in range(12)]
    random.Random(5701).shuffle(groups)
    split_of = {g: ("train" if i < 8 else "dev" if i < 10 else "test")
                for i, g in enumerate(groups)}
    tasks, records = [], []
    for i in range(12):
        a, b = i + 2, i * 3 + 5
        answer = a + b
        gid = f"source-{i:02}"
        for variant in range(2):
            tid = f"{gid}-v{variant}"
            prompt = (f"计算 {a}+{b}。" if variant == 0 else f"求 {a} 与 {b} 的和。")
            prompt += '只输出 JSON，answer 字段是数值字符串。'
            task = {"task_id": tid, "group_id": gid, "split": split_of[gid],
                    "prompt": prompt, "expected": str(answer),
                    "planned_ids": [f"{tid}-c{j}" for j in range(8)]}
            tasks.append(task)
            outputs = [canonical({"answer": str(answer)}),
                       canonical({"answer": f"{answer * 2}/2"}),
                       canonical({"answer": str(answer + 1)}),
                       canonical({"answer": str(answer + 2)}),
                       f"答案是{answer}", canonical({"answer": str(answer)}), ""]
            for j, output in enumerate(outputs):
                records.append({"task_id": tid, "sample_id": f"{tid}-c{j}",
                                "status": "error" if j == 6 else "ok",
                                "stop_reason": "length" if j == 5 else "eos" if j < 6 else "error",
                                "output": output, "producer": "handwritten_fixture"})
            # c7 没有记录：保留计划位置，不能伪造成空的 rejected。
    contract = {"version": VERSION, "grader_version": GRADER_VERSION,
                "label_source": "deterministic verifier, not human preference",
                "split_of": split_of, "max_pairs_per_prompt": 1,
                "tie_policy": "skip equal scores; stable first strict pair",
                "invalid_format_policy": "exclude in this lesson, not universally required",
                "task_digest": digest(tasks), "record_digest": digest(records)}
    return tasks, records, contract


def validate_tasks(tasks: list[dict]) -> None:
    require(type(tasks) is list and len(tasks) > 0, "需要非空题目列表")
    ids, sample_ids, group_splits, prompt_splits = set(), set(), {}, {}
    for task in tasks:
        require(task["task_id"] not in ids, "重复题目ID")
        ids.add(task["task_id"])
        require(task["split"] in {"train", "dev", "test"}, "未知划分")
        gid, split = task["group_id"], task["split"]
        require(group_splits.get(gid, split) == split, "同一来源跨划分")
        group_splits[gid] = split
        ph = digest(task["prompt"])
        require(prompt_splits.get(ph, split) == split, "相同提示跨划分")
        prompt_splits[ph] = split
        require(len(task["planned_ids"]) == len(set(task["planned_ids"])), "重复计划位置")
        for sid in task["planned_ids"]:
            require(sid not in sample_ids, "全局样本ID重复")
            sample_ids.add(sid)


def build_preferences(tasks: list[dict], records: list[dict]) -> dict:
    """一题最多保留一对；保留原始回答、评分身份和跳过原因，不自动补抽。"""
    validate_tasks(tasks)
    task_map = {t["task_id"]: t for t in tasks}
    by_id = {}
    for rec in records:
        tid, sid = rec["task_id"], rec["sample_id"]
        require(tid in task_map and sid in task_map[tid]["planned_ids"], "未知候选位置")
        require(sid not in by_id, "重复候选记录")
        by_id[sid] = rec
    pairs, audit, candidate_rows = [], [], []
    for task in tasks:
        graded = []
        for sid in task["planned_ids"]:
            rec = by_id.get(sid)
            grade = grade_candidate(task, rec)
            candidate_rows.append({"task_id": task["task_id"], "sample_id": sid, **grade})
            if grade["eligible"]:
                graded.append((rec, grade))
        edges = []
        ties = 0
        for (ra, ga), (rb, gb) in itertools.combinations(graded, 2):
            if ga["score"] == gb["score"]:
                ties += 1
                continue
            c, cg, r, rg = (ra, ga, rb, gb) if ga["score"] > gb["score"] else (rb, gb, ra, ga)
            if c["output"] == r["output"]:
                raise LessonError("相同回答不能构成严格偏好")
            edges.append((c, cg, r, rg))
        audit.append({"task_id": task["task_id"], "eligible_candidates": len(graded),
                      "tie_pairs_skipped": ties, "strict_pairs_available": len(edges),
                      "pairs_retained": int(bool(edges)),
                      "reason": "retained_one" if edges else "no_strict_preference"})
        if edges:
            c, cg, r, rg = edges[0]
            pairs.append({"pair_id": task["task_id"] + "-pair0", "group_id": task["group_id"],
                          "task_id": task["task_id"], "split": task["split"],
                          "prompt": task["prompt"], "chosen": c["output"], "rejected": r["output"],
                          "chosen_id": c["sample_id"], "rejected_id": r["sample_id"],
                          "chosen_score": cg["score"], "rejected_score": rg["score"],
                          "grader_version": GRADER_VERSION, "prompt_digest": digest(task["prompt"]),
                          "chosen_digest": digest(c), "rejected_digest": digest(r),
                          "label_source": "rule_verifier"})
    summary = {"planned_slots": sum(len(t["planned_ids"]) for t in tasks),
               "received_records": len(records), "pairs": len(pairs),
               "pairs_by_split": dict(Counter(p["split"] for p in pairs)),
               "states": dict(Counter(g["state"] for g in candidate_rows)),
               "all_strict_pairs_before_cap": sum(a["strict_pairs_available"] for a in audit),
               "ties_skipped": sum(a["tie_pairs_skipped"] for a in audit)}
    return {"pairs": pairs, "audit": audit, "candidate_grades": candidate_rows, "summary": summary}

In [4]:
tasks, records, data_contract = make_fixture()
data = build_preferences(tasks, records)
print(json.dumps(data["summary"], ensure_ascii=False, indent=2))
first_pair = next(p for p in data["pairs"] if p["split"] == "train")
print("一条训练偏好：")
print(json.dumps({k: first_pair[k] for k in
    ["task_id", "prompt", "chosen", "rejected", "label_source"]}, ensure_ascii=False, indent=2))

{
  "planned_slots": 192,
  "received_records": 168,
  "pairs": 24,
  "pairs_by_split": {
    "train": 16,
    "dev": 4,
    "test": 4
  },
  "states": {
    "graded": 96,
    "invalid_format": 24,
    "incomplete": 24,
    "generation_error": 24,
    "missing": 24
  },
  "all_strict_pairs_before_cap": 96,
  "ties_skipped": 48
}
一条训练偏好：
{
  "task_id": "source-00-v0",
  "prompt": "计算 2+5。只输出 JSON，answer 字段是数值字符串。",
  "chosen": "{\"answer\":\"7\"}",
  "rejected": "{\"answer\":\"8\"}",
  "label_source": "rule_verifier"
}


### 2.1 先按来源切分，再形成偏好

12个来源，每个来源两种表述。固定为8个训练来源、2个开发来源、2个测试来源，再评分与配对。

每题计划8个候选：两条等价正确、两条合法错误、一条格式错误、一条截断、一条运行错误、一条缺失。实际只有7条原始记录。

一题可形成4个严格组合，但不等于4个独立问题。本课每题只保留一对，防止候选多的问题隐式获得更大权重。选择顺序事先固定，不看测试成绩。

保留来源、评分器版本和原始记录摘要。`generation_requests.jsonl` 不含参考答案；`tasks_grader_only.json` 属于评分侧。所有课堂源码公开，不声称保密评测。

In [5]:
t = tasks[0]
rows = {r["sample_id"]: r for r in records if r["task_id"] == t["task_id"]}
for sid in t["planned_ids"]:
    g = grade_candidate(t, rows.get(sid))
    print(sid.rsplit("-",1)[-1], g["state"], "score=", g["score"], "用于偏好=", g["eligible"])
print("该题配对审计：", data["audit"][0])

c0 graded score= 1.0 用于偏好= True
c1 graded score= 1.0 用于偏好= True
c2 graded score= 0.0 用于偏好= True
c3 graded score= 0.0 用于偏好= True
c4 invalid_format score= None 用于偏好= False
c5 incomplete score= None 用于偏好= False
c6 generation_error score= None 用于偏好= False
c7 missing score= None 用于偏好= False
该题配对审计： {'task_id': 'source-00-v0', 'eligible_candidates': 4, 'tie_pairs_skipped': 2, 'strict_pairs_available': 4, 'pairs_retained': 1, 'reason': 'retained_one'}


### 2.2 奖励来源要明确，平分与不确定也要保留

数值正确、格式合规、工具执行成功、用户主观偏好，是不同维度。本课标签来自规则验证器，**不是人类偏好标注**。

人类对风格、简洁、详细程度可能有不同要求，分歧不总是错误。需要记录标准和来源，不应把所有平分随意标成 chosen/rejected。

下面的二维奖励模型使用另行构造的人工特征，**不会把本节的中文字符串冒充成已经学习好的语义向量**。

## 3. 奖励模型的成对损失

评分器给两份回答输出 $r_c,r_r$。Bradley–Terry 模型写成：

\[
P(c\succ r\mid x)=\sigma(r_c-r_r)
\]

对一条偏好记录，损失为：

\[
L_{RM}=-\log\sigma(r_c-r_r)=\operatorname{softplus}(r_r-r_c)
\]

对batch取平均，更新的是**评分模型**。[奖励模型文档](https://huggingface.co/docs/trl/reward_trainer)

分数不需要位于0到1。sigmoid表示这个偏好模型中的成对胜率，不是某条回答独立的正确概率，也不自动获得概率校准保证。

In [6]:
def finite_vector(t: torch.Tensor, name: str) -> None:
    require(isinstance(t, torch.Tensor) and t.is_floating_point(), f"{name} 必须是浮点张量")
    require(t.ndim == 1 and t.numel() > 0, f"{name} 必须是非空一维张量")
    require(bool(torch.isfinite(t).all()), f"{name} 必须有限")


def aligned_vectors(*tensors: torch.Tensor) -> None:
    for i, t in enumerate(tensors):
        finite_vector(t, f"t{i}")
        require(t.shape == tensors[0].shape and t.device == tensors[0].device
                and t.dtype == tensors[0].dtype, "向量形状/设备/精度必须一致，禁止隐式广播")


def pairwise_reward_loss(chosen_scores: torch.Tensor, rejected_scores: torch.Tensor) -> torch.Tensor:
    aligned_vectors(chosen_scores, rejected_scores)
    # -log(sigmoid(delta)) 的稳定写法。
    return F.softplus(rejected_scores - chosen_scores).mean()


def dpo_loss(policy_chosen: torch.Tensor, policy_rejected: torch.Tensor,
             reference_chosen: torch.Tensor, reference_rejected: torch.Tensor,
             beta: float = 0.5) -> torch.Tensor:
    aligned_vectors(policy_chosen, policy_rejected, reference_chosen, reference_rejected)
    require(type(beta) in {float, int} and math.isfinite(beta) and beta > 0,
            "beta 必须是有限正数")
    # reference 是固定的比较基线；本课主动阻止梯度进入它。
    delta = (policy_chosen - policy_rejected
             - reference_chosen.detach() + reference_rejected.detach())
    return F.softplus(-float(beta) * delta).mean()


def completion_logps(logits: torch.Tensor, input_ids: torch.Tensor,
                     completion_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """同位 token IDs 与回答掩码；在此只移位一次；返回每条回答的 log 概率之和。"""
    require(isinstance(logits, torch.Tensor) and logits.ndim == 3
            and logits.is_floating_point(), "logits 必须是[B,T,V]浮点张量")
    b, t, v = logits.shape
    require(b > 0 and t >= 2 and v >= 2, "张量维度过小")
    require(input_ids.shape == (b, t) and input_ids.dtype == torch.long, "ID形状或类型错误")
    require(completion_mask.shape == (b, t) and completion_mask.dtype == torch.bool,
            "completion_mask 必须是同位bool掩码")
    require(input_ids.device == logits.device == completion_mask.device, "设备不一致")
    require(bool(torch.isfinite(logits).all()), "logits含非有限值")
    require(bool(((input_ids >= 0) & (input_ids < v)).all()), "编号越界")
    require(not bool(completion_mask[:, 0].any()), "位置0没有可用的前一位置预测")
    mask = completion_mask[:, 1:]
    counts = mask.sum(dim=-1)
    require(bool((counts > 0).all()), "每条回答都必须有至少一个有效预测目标")
    # FP16/BF16 时先用FP32稳定计算；本课实验用FP64。
    work = logits if logits.dtype in (torch.float32, torch.float64) else logits.float()
    log_probs = F.log_softmax(work[:, :-1, :], dim=-1)
    token_logps = log_probs.gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    return (token_logps * mask).sum(dim=-1), counts


def normalized_advantages(rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    finite_vector(rewards, "rewards")
    require(rewards.numel() >= 2, "相对优势至少需要两个样本")
    require(type(eps) in {float, int} and math.isfinite(eps) and eps > 0, "eps错误")
    centered = rewards.detach() - rewards.detach().mean()
    return centered / (rewards.detach().std(unbiased=False) + eps)


def regularized_objective(logits: torch.Tensor, reference_logps: torch.Tensor,
                          rewards: torch.Tensor, kl_weight: float = 0.3) -> torch.Tensor:
    """返回 loss=-E[r]+lambda*KL；遍历全部有限动作的精确期望，不是PPO/GRPO。"""
    aligned_vectors(logits, reference_logps, rewards)
    require(type(kl_weight) in {float, int} and math.isfinite(kl_weight) and kl_weight > 0,
            "KL权重必须为有限正数")
    require(abs(float(reference_logps.detach().logsumexp(0))) < 1e-7, "reference 不是正规化log概率")
    logp = F.log_softmax(logits, dim=0)
    p = logp.exp()
    return -(p * rewards.detach()).sum() + kl_weight * (p * (logp - reference_logps.detach())).sum()

In [7]:
cs = torch.tensor([1.0,2.0], dtype=DTYPE, requires_grad=True)
rs = torch.tensor([0.0,1.0], dtype=DTYPE, requires_grad=True)
loss = pairwise_reward_loss(cs,rs)
loss.backward()
print("loss:",loss.item())
print("chosen梯度:",cs.grad.tolist())
print("rejected梯度:",rs.grad.tolist())
print("两侧加100后的loss:",pairwise_reward_loss(cs.detach()+100,rs.detach()+100).item())

loss: 0.31326168751822286
chosen梯度: [-0.13447071068499755, -0.13447071068499755]
rejected梯度: [0.13447071068499755, 0.13447071068499755]
两侧加100后的loss: 0.31326168751822286


### 3.1 整体加100，偏好不变

差值不变，损失与成对概率也不变。同一提示下加入与候选无关的偏置，可以在差值中抵消。

因此，不能仅根据原始分数100比0高，就声称“事实更正确”；不同奖励模型的原始分数也不天然共用同一尺度。

本例chosen梯度为负，梯度下降推动它上升；rejected梯度为正，推动它下降。共享参数的复杂网络并不保证每个样本在每一步都这样变化。

实现使用稳定的 `softplus` / `logsigmoid`，不先sigmoid再对可能下溢为0的结果取log。[PyTorch LogSigmoid](https://docs.pytorch.org/docs/2.10/generated/torch.nn.functional.logsigmoid.html)

In [8]:
print("同分loss:",pairwise_reward_loss(torch.zeros(1),torch.zeros(1)).item())
print("极端错误排序:",pairwise_reward_loss(torch.tensor([-1000.]),torch.tensor([1000.])).item())

同分loss: 0.6931471824645996
极端错误排序: 2000.0


## 4. 训练一个真正的小评分器：观察数据混杂

\[
r_\phi(z)=w_0z_0+w_1z_1
\]

$z_0$ 是与人工偏好规则相关的特征坐标，$z_1$ 是中心化的长度代理。**这不是从真实文本抽取的语义，也不表示文本长度可以是负数。**

| 数据集 | 特征关系 |
|---|---|
| `confounded` 混杂 | chosen的$z_0$更高，$z_1$总与它同向，幅度为3倍 |
| `balanced` 平衡 | 同样$z_0$下，长度代理的正反方向成对出现 |

两组均为32对、零初始化、SGD训练120步。固定16对留出数据把长度相关性反转，不参与优化或轮次选择。

In [9]:
def make_reward_features(kind: str, n: int = 32) -> tuple[torch.Tensor, torch.Tensor]:
    require(kind in {"confounded", "balanced", "shifted"}, "未知特征集")
    require(type(n) is int and n >= 2 and n % 2 == 0, "需要偶数个特征对")
    signal = torch.linspace(0.6, 1.4, n, dtype=DTYPE)
    if kind == "balanced":
        # 同一信号强度下，长度正负两种方向严格配对。
        signal = torch.linspace(0.6, 1.4, n // 2, dtype=DTYPE).repeat_interleave(2)
        sign = torch.tensor([1., -1.] * (n // 2), dtype=DTYPE)
    elif kind == "shifted":
        signal = torch.linspace(0.65, 1.35, n, dtype=DTYPE)
        sign = -torch.ones(n, dtype=DTYPE)
    else:
        sign = torch.ones(n, dtype=DTYPE)
    chosen = torch.stack([signal, 3 * signal * sign], dim=1)
    rejected = -chosen
    return chosen, rejected


def evaluate_ranker(model: nn.Module, kind: str) -> dict:
    c, r = make_reward_features(kind, 16)
    with torch.no_grad():
        cs, rs = model(c).squeeze(-1), model(r).squeeze(-1)
        margins = cs - rs
        return {"pairs": len(c), "loss": float(pairwise_reward_loss(cs, rs)),
                "pair_accuracy": float((margins > 0).double().mean()),
                "min_margin": float(margins.min())}


def train_reward_experiment(steps: int = 120) -> tuple[dict, dict]:
    """x0是人工特征坐标，x1是长度代理；不从真实文本抽取、不读取测试标签训练。"""
    results, states = {}, {}
    # 这个独立生成分布从不加入RM优化器，用哈希确认奖励训练未改生成器。
    generator_logits = nn.Parameter(torch.tensor([0.2, 0.5, 0.3], dtype=DTYPE).log())
    generator_before = tensor_digest(generator_logits)
    for kind in ["confounded", "balanced"]:
        model = nn.Linear(2, 1, bias=False).to(dtype=DTYPE)
        with torch.no_grad():
            model.weight.zero_()
        c, r = make_reward_features(kind)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.15)
        history = []
        for step in range(steps + 1):
            cs, rs = model(c).squeeze(-1), model(r).squeeze(-1)
            loss = pairwise_reward_loss(cs, rs)
            if step % 20 == 0 or step == steps:
                history.append({"step": step, "loss": float(loss.detach()),
                                "pair_accuracy": float((cs.detach() > rs.detach()).double().mean())})
            if step == steps:
                break
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        results[kind] = {"weights": model.weight.detach().flatten().tolist(),
                         "train": {"loss": float(loss.detach()),
                                   "pair_accuracy": float((cs > rs).double().mean())},
                         "shifted_test": evaluate_ranker(model, "shifted"), "history": history}
        states[kind] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    results["contract"] = {"parameters_each": 2, "train_pairs_each": 32, "test_pairs": 16,
                           "steps": steps, "lr": 0.15, "optimizer": "SGD",
                           "features": ["synthetic task-relevant coordinate", "synthetic verbosity proxy"],
                           "generation_policy_unchanged": tensor_digest(generator_logits) == generator_before,
                           "scope": "刻意构造的特征偏移，不是自然语言奖励模型泛化成绩"}
    return results, states

In [10]:
reward_report, reward_states = train_reward_experiment()
for name in ["confounded", "balanced"]:
    r=reward_report[name]
    print(name, "权重",r["weights"],
          "训练排序",r["train"]["pair_accuracy"],
          "反相关留出排序",r["shifted_test"]["pair_accuracy"])
print("生成策略参数保持不变：",reward_report["contract"]["generation_policy_unchanged"])

confounded 权重 [0.36108846691555097, 1.0832654007466522] 训练排序 1.0 反相关留出排序 0.0
balanced 权重 [2.1817646946190723, 0.0] 训练排序 1.0 反相关留出排序 1.0
生成策略参数保持不变： True


### 4.1 训练全对，仍然可能学错依据

混杂模型更依赖幅度更大的长度代理：训练排序全对，反相关留出全错。平衡构造使长度权重保持为零，能通过本课固定条件。

这不保证“加几对反例就解决真实RM泛化”，也不是自然语言benchmark。它只验证：**高训练排序准确率不能识别所有混杂**。

另一个独立生成分布的参数哈希保持不变，说明本阶段只训练了评分模型。奖励模型变好，不等于生成器已经会产生更好的回答。

奖励模型是不完美代理，优化它也可能损害真实目标；研究分别观察过RL和Best-of-N下的过优化。[奖励过优化论文](https://arxiv.org/abs/2210.10760)

## 5. “得到1分”怎样接到生成模型梯度？

把评分数字转成可求导张量，再执行 backward，不能凭空连接到生成模型。适当条件下，策略梯度满足：

\[
\nabla_\theta\mathbb E_{y\sim\pi_\theta}[r(y)]
=\mathbb E_{y\sim\pi_\theta}[r(y)\nabla_\theta\log\pi_\theta(y)]
\]

奖励可以来自不可导的离散测试；梯度通过**生成该回答的log概率**传回策略。下方只演示固定动作0的单样本梯度，不是完整采样训练或PPO/GRPO。

In [11]:
def gradient_signal_demo() -> dict:
    """评分可不可导，与策略是否收到梯度是两个问题；固定动作0用于手算。"""
    logits = nn.Parameter(torch.zeros(2, dtype=DTYPE))
    detached_score = torch.tensor(1.0, dtype=DTYPE, requires_grad=True)
    (-detached_score).backward()
    disconnected = logits.grad is None
    logp = F.log_softmax(logits, dim=0)
    # 这里是单个固定样本的策略梯度形式，不是完整on-policy训练实验。
    score = torch.tensor(1.0, dtype=DTYPE)
    loss = -score.detach() * logp[0]
    loss.backward()
    return {"backprop_only_score_leaves_policy_grad_none": disconnected,
            "logprob_weighted_policy_gradient": logits.grad.tolist(),
            "scope": "固定动作0的单样本梯度示意；无rollout、无PPO/GRPO"}


def objective_mechanisms() -> dict:
    chosen = torch.tensor([1.0, 2.0], dtype=DTYPE, requires_grad=True)
    rejected = torch.tensor([0.0, 1.0], dtype=DTYPE, requires_grad=True)
    loss = pairwise_reward_loss(chosen, rejected)
    loss.backward()
    shifted = pairwise_reward_loss(chosen.detach() + 100, rejected.detach() + 100)
    p0 = torch.tensor([0.4, 0.4, 0.2], dtype=DTYPE)
    p1 = torch.tensor([0.3, 0.1, 0.6], dtype=DTYPE)
    ref = p0.log()
    def dl(p):
        l = p.log()
        return float(dpo_loss(l[:1], l[1:2], ref[:1], ref[1:2], beta=0.5))
    # 每个token的条件概率：[0.8,0.5] vs [0.6]。
    sum_logps = [math.log(0.8) + math.log(0.5), math.log(0.6)]
    mean_logps = [sum_logps[0] / 2, sum_logps[1]]
    proxy = torch.tensor([0.2, 1.0], dtype=DTYPE)
    truth = torch.tensor([1.0, 0.0], dtype=DTYPE)
    base_logps = torch.tensor([0.5, 0.5], dtype=DTYPE).log()
    proxy_rows = []
    for lam in [1.0, 0.3, 0.1]:
        optimal = F.softmax(base_logps + proxy / lam, dim=0)
        proxy_rows.append({"kl_weight": lam, "probabilities": optimal.tolist(),
                           "expected_proxy": float((optimal * proxy).sum()),
                           "expected_true_utility": float((optimal * truth).sum())})
    return {
        "gradient_path": gradient_signal_demo(),
        "pairwise": {"loss": float(loss.detach()), "shifted_loss": float(shifted),
                     "chosen_grad": chosen.grad.tolist(), "rejected_grad": rejected.grad.tolist(),
                     "equal_scores_loss": math.log(2)},
        "dpo_relative_counterexample": {"reference": p0.tolist(), "new_policy": p1.tolist(),
                                        "initial_loss": dl(p0), "new_loss": dl(p1),
                                        "chosen_probability_fell": True},
        "length": {"sum_logps": sum_logps, "mean_logps": mean_logps,
                   "sum_prefers": "short_B", "mean_prefers": "long_A"},
        "group_advantages": {"mixed": normalized_advantages(torch.tensor([1., 0., 1., 0.], dtype=DTYPE)).tolist(),
                             "all_equal": normalized_advantages(torch.ones(4, dtype=DTYPE)).tolist(),
                             "scope": "仅组内优势归一化，不是完整GRPO"},
        "proxy_optimum": proxy_rows,
        "boundary": "人工有限动作反例，不是大模型、无语言能力结论",
    }

In [12]:
mechanisms=objective_mechanisms()
print(json.dumps(mechanisms["gradient_path"],ensure_ascii=False,indent=2))
print("固定样本loss=-log p(A)，初始两动作概率各0.5。")
print("梯度[-0.5,0.5]在梯度下降时提高A相对B的logit。")

{
  "backprop_only_score_leaves_policy_grad_none": true,
  "logprob_weighted_policy_gradient": [
    -0.5,
    0.5
  ],
  "scope": "固定动作0的单样本梯度示意；无rollout、无PPO/GRPO"
}
固定样本loss=-log p(A)，初始两动作概率各0.5。
梯度[-0.5,0.5]在梯度下降时提高A相对B的logit。


## 6. 序列log概率：接回语言模型时的关键

\[
\log\pi_\theta(y\mid x)=\sum_t\log\pi_\theta(y_t\mid x,y_{<t})
\]

只统计回答token与约定结束符，不统计提示或padding。输入和mask同位，在本函数中只移位一次：

```python
log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
next_ids = input_ids[:, 1:]
next_mask = completion_mask[:, 1:]
```

`log_softmax`提供更稳定的输出和梯度，不必先softmax再log。[PyTorch文档](https://docs.pytorch.org/docs/2.10/generated/torch.nn.functional.log_softmax.html)

**mask控制计分，不是把前文从attention里删掉。** 模型仍需正确的因果掩码，本函数不能修复读取未来的错误。

In [13]:
ids=torch.tensor([[0,1,2,3,0],[0,2,1,0,0]],dtype=torch.long)
mask=torch.tensor([[0,0,1,1,0],[0,1,1,1,0]],dtype=torch.bool)
logits=torch.zeros(2,5,4,dtype=DTYPE,requires_grad=True)
sums,counts=completion_logps(logits,ids,mask)
print("有效目标:",counts.tolist())
print("log概率和:",sums.detach().tolist())
print("手算:",[-2*math.log(4),-3*math.log(4)])
sums.sum().backward()
print("样本0位置0梯度:",logits.grad[0,0].tolist())
print("样本0位置1梯度:",logits.grad[0,1].tolist())

有效目标: [2, 3]
log概率和: [-2.772588722239781, -4.1588830833596715]
手算: [-2.772588722239781, -4.1588830833596715]
样本0位置0梯度: [0.0, 0.0, 0.0, 0.0]
样本0位置1梯度: [-0.25, -0.25, 0.75, -0.25]


### 6.1 求和不能悄悄换成均值

回答A有两个token，条件概率分别0.8和0.5，序列概率为0.4；回答B有一个token，条件概率为0.6。

按序列概率，B更高；按平均token log概率，A可能更高。这是换了目标，不只是“归一化一下”。

原始sigmoid DPO使用序列log概率；长度归一化是另有约定的变体，不能宣称完全等价。[DPO说明](https://huggingface.co/docs/trl/dpo_trainer)

In [14]:
print(json.dumps(mechanisms["length"],ensure_ascii=False,indent=2))

{
  "sum_logps": [
    -0.916290731874155,
    -0.5108256237659907
  ],
  "mean_logps": [
    -0.4581453659370775,
    -0.5108256237659907
  ],
  "sum_prefers": "short_B",
  "mean_prefers": "long_A"
}


## 7. 三种策略训练目标

### 7.1 SFT：模仿示范

\[
L_{SFT}=-\log\pi_\theta(y^+|x)
\]

这一项直接学习chosen示范，不使用rejected。

### 7.2 DPO：相对固定参考，改善成对差距

\[
\Delta_\theta=(\log\pi_\theta(y^+|x)-\log\pi_\theta(y^-|x))
-(\log\pi_{ref}(y^+|x)-\log\pi_{ref}(y^-|x))
\]

\[
L_{DPO}=-\log\sigma(\beta\Delta_\theta)
\]

reference固定，policy接受梯度。原始DPO不要求额外训练显式奖励模型。[原论文](https://arxiv.org/abs/2305.18290)

$\beta$在推导中与KL正则相关，在损失里也控制缩放与梯度。它不是学习率，也不能简单推断增大它必然让每次更新的KL如何变化。

本课主动detach参考log概率，并检查参考参数无梯度。真实大模型还需正确组织冻结reference、模板、分词和适配器状态。

In [15]:
pc=torch.tensor([-1.],dtype=DTYPE,requires_grad=True)
pr=torch.tensor([-2.],dtype=DTYPE,requires_grad=True)
rc=torch.tensor([-3.],dtype=DTYPE,requires_grad=True)
rr=torch.tensor([-1.],dtype=DTYPE,requires_grad=True)
loss=dpo_loss(pc,pr,rc,rr,beta=.5)
loss.backward()
print("loss:",loss.item())
print("policy梯度:",pc.grad.item(),pr.grad.item())
print("reference梯度:",rc.grad,rr.grad)

loss: 0.2014132779827524
policy梯度: -0.09121276190317816 0.09121276190317816
reference梯度: None None


### 7.3 奖励加KL：精确期望不冒充RL算法

有限动作可以全部枚举，本课最小化：

\[
L=-\sum_y\pi_\theta(y)r(y)
+\lambda\sum_y\pi_\theta(y)\log\frac{\pi_\theta(y)}{\pi_{ref}(y)}
\]

第一项鼓励高奖励，第二项约束偏离参考。本课遍历全部三个动作，可以直接微分期望。

**这不是PPO/GRPO。** 真实语言模型不能枚举所有回答，还需处理采样、方差、旧策略与其他更新约束。

固定奖励、参考概率为正时，最优解为：

\[
\pi^*(y)\propto\pi_{ref}(y)e^{r(y)/\lambda}
\]

用它核对数值优化结果，不能因此推断真实模型后训练方法的排名。

In [16]:
def train_policy_experiment(steps: int = 160) -> tuple[dict, dict]:
    """3个固定回答视为3个原子动作；3个logits参数，不学习语言或生成新答案。"""
    reference = torch.tensor([0.2, 0.5, 0.3], dtype=DTYPE)
    reference_logps = reference.log().detach().clone()
    ref_hash = tensor_digest(reference_logps)
    utility = torch.tensor([1.0, 0.0, 0.2], dtype=DTYPE)
    results, states = {}, {}
    for method in ["sft", "dpo", "exact_reward_kl"]:
        logits = nn.Parameter(reference_logps.clone())
        optimizer = torch.optim.Adam([logits], lr=0.04)
        history = []
        for step in range(steps + 1):
            logp = F.log_softmax(logits, dim=0)
            if method == "sft":
                loss = -logp[0]
            elif method == "dpo":
                loss = dpo_loss(logp[:1], logp[1:2], reference_logps[:1], reference_logps[1:2], beta=0.5)
            else:
                loss = regularized_objective(logits, reference_logps, utility, kl_weight=0.3)
            if step % 40 == 0 or step == steps:
                history.append({"step": step, "own_objective": float(loss.detach()),
                                "probabilities": logp.detach().exp().tolist()})
            if step == steps:
                break
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        p = F.softmax(logits.detach(), dim=0)
        results[method] = {"probabilities": p.tolist(), "own_objective": float(loss.detach()),
                           "expected_known_utility": float((p * utility).sum()),
                           "kl_to_reference": float((p * (p.log() - reference_logps)).sum()),
                           "history": history}
        states[method] = logits.detach().clone()
    analytic = F.softmax(reference_logps + utility / 0.3, dim=0)
    results["contract"] = {"initial_probabilities": reference.tolist(), "actions": ["A: preferred", "B: rejected", "C: unpaired"],
                           "steps": steps, "trainable_parameters_each": 3,
                           "reference_frozen": tensor_digest(reference_logps) == ref_hash,
                           "reference_grad": None,
                           "reward_optimum": analytic.tolist(),
                           "exact_reward_max_error": float((states["exact_reward_kl"].softmax(0) - analytic).abs().max()),
                           "boundary": "三种目标不是同一个标尺；无独立泛化评测；精确期望不等于PPO/GRPO"}
    return results, {"reference_logps": reference_logps, **states}

In [17]:
policy_report,policy_states=train_policy_experiment()
print("初始分布:",policy_report["contract"]["initial_probabilities"])
for name in ["sft","dpo","exact_reward_kl"]:
    print(name,[round(v,6) for v in policy_report[name]["probabilities"]])
print("参考未更新:",policy_report["contract"]["reference_frozen"])
print("精确奖励目标与解析最优分布的误差:",policy_report["contract"]["exact_reward_max_error"])

初始分布: [0.2, 0.5, 0.3]
sft [0.973351, 0.016656, 0.009993]
dpo [0.942541, 0.00344, 0.054019]
exact_reward_kl [0.837882, 0.074765, 0.087353]
参考未更新: True
精确奖励目标与解析最优分布的误差: 5.301629684117959e-05


### 7.4 不能把三个动作表当成语言能力测量

A、B、C只是三个固定回答的原子编号，初始概率`[0.2,0.5,0.3]`。

SFT只示范A；DPO只拿到A优于B；精确奖励目标得到全部奖励`[1,0,0.2]`，并用KL权重0.3。都训练160步，每个策略只有三个logits参数。

这里没有文本编码器、不会创造新答案，也没有独立语言测试集。三种目标的监督信息和约束不同，不因SFT给A的概率最高就宣称SFT更强。各自的loss也不是可直接横向排名的共同量尺。

## 8. DPO下降，chosen的绝对概率仍然可能下降

参考分布`[0.4,0.4,0.2]`，新分布`[0.3,0.1,0.6]`。

A从0.4降为0.3，但A/B从1变成3，相对偏好差距改善。因此DPO loss可以下降。

下方只是两组合法分布的目标比较，不声称它是上一训练轨迹中的某一步。

In [18]:
print(json.dumps(mechanisms["dpo_relative_counterexample"],ensure_ascii=False,indent=2))

{
  "reference": [
    0.4,
    0.4,
    0.2
  ],
  "new_policy": [
    0.3,
    0.1,
    0.6
  ],
  "initial_loss": 0.6931471805599453,
  "new_loss": 0.4557463944083263,
  "chosen_probability_fell": true
}


需要同时观察偏好margin、各自log概率、与参考的KL、任务评分、旧能力与生成成本，而不是仅看训练loss。

本课还令真实效用为`[1,0]`，代理却给`[0.2,1]`。减少KL约束、精确求解代理目标后，代理分数变高，真实效用反而降低。**这是假设构造，不是实测LLM退化幅度。**[代理过优化](https://arxiv.org/abs/2210.10760)

In [19]:
for row in mechanisms["proxy_optimum"]:
    print("KL权重",row["kl_weight"],"代理",round(row["expected_proxy"],6),
          "真实效用",round(row["expected_true_utility"],6))

KL权重 1.0 代理 0.75198 真实效用 0.310026
KL权重 0.3 代理 0.948025 真实效用 0.064969
KL权重 0.1 代理 0.999732 真实效用 0.000335


## 9. 为GRPO准备：只观察组内优势，不假装实现完整算法

本课定义：

\[
A_i=\frac{r_i-\overline r}{\operatorname{std}_{population}(r)+\epsilon}
\]

显式使用总体标准差`unbiased=False`。具体算法可能有不同归一化约定，需要查实现。

`[1,0,1,0]`得到接近`[1,-1,1,-1]`。全同分时中心化后为零，当前组没有相对奖励方向；完整目标中的KL等其他项仍可能产生更新。

GRPO还包含采样、策略比率、约束等步骤，这个单元只是一个组成部分。[DeepSeekMath论文](https://arxiv.org/abs/2402.03300)

In [20]:
print("混合奖励:",mechanisms["group_advantages"]["mixed"])
print("全部同分:",mechanisms["group_advantages"]["all_equal"])

混合奖励: [0.9999999800000003, -0.9999999800000003, 0.9999999800000003, -0.9999999800000003]
全部同分: [0.0, 0.0, 0.0, 0.0]


## 10. 独立练习

下一个实现单元提供检查函数，不包含练习参考答案。损失需要保持计算图，不应提前`.item()`或转Python浮点。

分别完成三题后再打开检查开关。不要直接复制下面后半部分的参考答案。

In [21]:
def close(actual, expected, atol: float = 1e-10) -> None:
    a = torch.as_tensor(actual, dtype=DTYPE)
    e = torch.as_tensor(expected, dtype=DTYPE)
    if a.shape != e.shape or not bool(torch.allclose(a, e, atol=atol, rtol=1e-8)):
        raise AssertionError(f"实际 {a.tolist()} != 预期 {e.tolist()}")


def check(ok, message: str = "检查失败") -> None:
    if not bool(ok):
        raise AssertionError(message)


def raises(call, error=LessonError):
    try:
        call()
    except error:
        return
    raise AssertionError(f"没有抛出 {error.__name__}")


def exercise_check_a(fn) -> None:
    c = torch.tensor([1., -2.], dtype=DTYPE, requires_grad=True)
    r = torch.tensor([0., 3.], dtype=DTYPE, requires_grad=True)
    actual = fn(c, r)
    expected = F.softplus(r - c).mean()
    close(actual, expected)
    gradients = torch.autograd.grad(actual, (c, r))
    wanted = torch.autograd.grad(expected, (c, r))
    for a, b in zip(gradients, wanted):
        close(a, b)
    close(fn(c.detach() + 91, r.detach() + 91), expected.detach())
    close(fn(torch.tensor([-1000.]), torch.tensor([1000.])), 2000.)


def exercise_check_b(fn) -> None:
    x = torch.zeros(2, 5, 4, dtype=DTYPE, requires_grad=True)
    ids = torch.tensor([[0, 1, 2, 3, 0], [0, 2, 1, 0, 0]], dtype=torch.long)
    mask = torch.tensor([[0, 0, 1, 1, 0], [0, 1, 1, 1, 0]], dtype=torch.bool)
    sums, counts = fn(x, ids, mask)
    close(sums, [-2 * math.log(4), -3 * math.log(4)])
    close(counts, [2, 3])
    sums.sum().backward()
    check(float(x.grad[0, 0].abs().sum()) == 0, "提示目标位置不应计分")
    check(float(x.grad[0, 1].abs().sum()) > 0, "第一枚答案由其前一位置预测")
    check(float(x.grad[:, -1].abs().sum()) == 0, "最后一个logits没有对应的下一token目标")
    modified = x.detach().clone()
    modified[0, 0, 0] = 1000
    modified[0, 3:, 1] = 1000
    close(fn(modified, ids, mask)[0], sums.detach())


def exercise_check_c(fn) -> None:
    pc = torch.tensor([-1.0], dtype=DTYPE, requires_grad=True)
    pr = torch.tensor([-2.0], dtype=DTYPE, requires_grad=True)
    rc = torch.tensor([-3.0], dtype=DTYPE, requires_grad=True)
    rr = torch.tensor([-1.0], dtype=DTYPE, requires_grad=True)
    loss = fn(pc, pr, rc, rr, 0.5)
    close(loss, F.softplus(torch.tensor(-1.5, dtype=DTYPE)))
    loss.backward()
    check(pc.grad.item() < 0 and pr.grad.item() > 0, "偏好梯度方向错误")
    check(rc.grad is None and rr.grad is None, "reference 必须冻结")
    close(fn(rc.detach(), rr.detach(), rc, rr, 0.5), math.log(2))
    check(float(fn(pc.detach(), pr.detach(), torch.tensor([1.], dtype=DTYPE), rr.detach(), 0.5)) > float(loss.detach()),
          "不能省略reference差值")


def make_tests() -> list[tuple[str, object]]:
    tests = []
    def add(name, fn):
        tests.append((name, fn))
    for a, b in [("1/2", "0.5"), ("-3/2", "-1.5"), ("28/2", "14"), ("0", "-0")]:
        add(f"有理数等价:{a}", lambda a=a, b=b: check(parse_number(a) == parse_number(b)))
    for raw in [True, "1/0", "nan", "1 or 2", "", "1e100", "9" * 65]:
        add(f"拒绝非法数值:{raw!r}", lambda raw=raw: raises(lambda: parse_number(raw)))
    for raw in ['{"a":1,"a":2}', '{"a":NaN}', '{', '[]\n{}']:
        add(f"严格JSON:{raw}", lambda raw=raw: raises(lambda: strict_json(raw)))
    tasks, records, _ = make_fixture()
    task = tasks[0]
    mine = [r for r in records if r["task_id"] == task["task_id"]]
    add("正确答案得分1", lambda: check(grade_candidate(task, mine[0])["score"] == 1))
    add("等价正确答案得分1", lambda: check(grade_candidate(task, mine[1])["score"] == 1))
    add("已完成错误答案得分0", lambda: check(grade_candidate(task, mine[2])["score"] == 0))
    for j, state in [(4, "invalid_format"), (5, "incomplete"), (6, "generation_error")]:
        add(f"不是反例:{state}", lambda j=j, state=state: check(
            grade_candidate(task, mine[j])["state"] == state and grade_candidate(task, mine[j])["score"] is None
            and not grade_candidate(task, mine[j])["eligible"]))
    add("缺失不是零分反例", lambda: check(grade_candidate(task, None)["score"] is None))
    wrong_task = copy.deepcopy(mine[0]); wrong_task["task_id"] = "other"
    add("回答不得串题", lambda: raises(lambda: grade_candidate(task, wrong_task)))
    payload = build_preferences(tasks, records)
    add("固定题量与来源", lambda: check(len(tasks) == 24 and len({t['group_id'] for t in tasks}) == 12))
    add("计划槽保留", lambda: check(payload["summary"]["planned_slots"] == 192 and len(records) == 168))
    add("每题只取一对", lambda: check(len(payload["pairs"]) == 24))
    add("来源先切分", lambda: check(payload["summary"]["pairs_by_split"] == {"train": 16, "dev": 4, "test": 4}))
    add("保留平分计数", lambda: check(payload["summary"]["ties_skipped"] == 48))
    add("二次组合不放大提示权重", lambda: check(payload["summary"]["all_strict_pairs_before_cap"] == 96))
    add("偏好方向和身份", lambda: check(all(p["chosen_score"] > p["rejected_score"]
        and p["grader_version"] == GRADER_VERSION and p["chosen"] != p["rejected"] for p in payload["pairs"])))
    add("相同答案不造反例", lambda: check(not build_preferences([task], mine[:2])["pairs"]))
    add("全失败不造反例", lambda: check(not build_preferences([task], mine[4:])["pairs"]))
    add("记录顺序不改变选择", lambda: check(build_preferences(tasks, list(reversed(records))) == payload))
    add("拒绝重复候选", lambda: raises(lambda: build_preferences(tasks, records + records[:1])))
    crossed = copy.deepcopy(tasks); crossed[1]["split"] = "test" if crossed[0]["split"] != "test" else "train"
    add("拒绝来源跨侧", lambda: raises(lambda: validate_tasks(crossed)))
    add("只更改参考不改变生成提示", lambda: check(digest([{k: t[k] for k in ['task_id','prompt']} for t in tasks]) ==
        digest([{k: t[k] for k in ['task_id','prompt']} for t in [dict(x, expected='999') for x in tasks]])))
    add("奖励损失与梯度", lambda: exercise_check_a(pairwise_reward_loss))
    add("相等分数损失log2", lambda: close(pairwise_reward_loss(torch.zeros(3), torch.zeros(3)), math.log(2), 1e-7))
    add("奖励分数不是概率", lambda: close(pairwise_reward_loss(torch.tensor([100.]), torch.tensor([100.])), math.log(2), 1e-7))
    add("拒绝奖励广播", lambda: raises(lambda: pairwise_reward_loss(torch.zeros(2), torch.zeros(1))))
    add("拒绝奖励NaN", lambda: raises(lambda: pairwise_reward_loss(torch.tensor([float('nan')]), torch.zeros(1))))
    add("拒绝空奖励", lambda: raises(lambda: pairwise_reward_loss(torch.zeros(0), torch.zeros(0))))
    add("DPO公式与reference梯度", lambda: exercise_check_c(dpo_loss))
    for beta in [0, -1, float('nan'), True]:
        add(f"拒绝beta:{beta}", lambda beta=beta: raises(lambda: dpo_loss(*[torch.zeros(1)] * 4, beta=beta)))
    add("序列log概率与梯度", lambda: exercise_check_b(completion_logps))
    lx = torch.zeros(1, 3, 4, dtype=DTYPE)
    ids = torch.tensor([[0, 1, 2]])
    add("拒绝全忽略回答", lambda: raises(lambda: completion_logps(lx, ids, torch.zeros(1,3,dtype=torch.bool))))
    add("拒绝位置0目标", lambda: raises(lambda: completion_logps(lx, ids, torch.ones(1,3,dtype=torch.bool))))
    add("拒绝越界编号", lambda: raises(lambda: completion_logps(lx, torch.tensor([[0,1,4]]), torch.tensor([[0,1,1]],dtype=torch.bool))))
    add("拒绝浮点ID", lambda: raises(lambda: completion_logps(lx, ids.float(), torch.tensor([[0,1,1]],dtype=torch.bool))))
    add("奖励相同优势全零", lambda: close(normalized_advantages(torch.ones(4, dtype=DTYPE)), [0]*4))
    add("组内优势方向", lambda: close(normalized_advantages(torch.tensor([1.,0.,1.,0.],dtype=DTYPE)), [1,-1,1,-1],1e-7))
    add("优势不反传到奖励", lambda: check(not normalized_advantages(torch.ones(4, requires_grad=True)).requires_grad))
    add("单样本不做标准化", lambda: raises(lambda: normalized_advantages(torch.ones(1))))
    add("固定reference期望KL为零", lambda: close(regularized_objective(torch.log(torch.tensor([.5,.5],dtype=DTYPE)),
        torch.log(torch.tensor([.5,.5],dtype=DTYPE)), torch.zeros(2,dtype=DTYPE)),0))
    m = objective_mechanisms()
    add("只对评分反传不能更新策略", lambda: check(m['gradient_path']['backprop_only_score_leaves_policy_grad_none']))
    add("策略梯度连接log概率", lambda: close(m['gradient_path']['logprob_weighted_policy_gradient'],[-.5,.5]))
    add("DPO下降不保证chosen绝对增加", lambda: check(m['dpo_relative_counterexample']['new_loss'] < m['dpo_relative_counterexample']['initial_loss']
        and m['dpo_relative_counterexample']['new_policy'][0] < m['dpo_relative_counterexample']['reference'][0]))
    add("求和与均值不是同一目标", lambda: check(m['length']['sum_logps'][0] < m['length']['sum_logps'][1]
        and m['length']['mean_logps'][0] > m['length']['mean_logps'][1]))
    add("代理优化可以损害真实目标", lambda: check(m['proxy_optimum'][-1]['expected_proxy'] > m['proxy_optimum'][0]['expected_proxy']
        and m['proxy_optimum'][-1]['expected_true_utility'] < m['proxy_optimum'][0]['expected_true_utility']))
    # 短训练用于机制检查，不以完整报告的精确浮点数写死答案。
    rm, rm_states = train_reward_experiment(40)
    add("奖励模型训练集可拟合", lambda: check(rm['confounded']['train']['pair_accuracy'] == 1))
    add("混杂模型在反向相关集失败", lambda: check(rm['confounded']['shifted_test']['pair_accuracy'] == 0))
    add("平衡模型在固定反向相关集通过", lambda: check(rm['balanced']['shifted_test']['pair_accuracy'] == 1))
    add("RM训练未修改生成参数", lambda: check(rm['contract']['generation_policy_unchanged']))
    pp, pp_states = train_policy_experiment(60)
    for name in ['sft','dpo','exact_reward_kl']:
        add(f"策略发生更新:{name}", lambda name=name: check(pp[name]['probabilities'][0] > .2))
        add(f"策略正规化:{name}", lambda name=name: close(sum(pp[name]['probabilities']),1))
    add("reference保持冻结", lambda: check(pp['contract']['reference_frozen']))
    def save_reload_test():
        with tempfile.TemporaryDirectory() as temp:
            path = Path(temp)/'weights.pt'
            torch.save({'rm':rm_states, 'policy':pp_states},path)
            got = torch.load(path, map_location='cpu', weights_only=True)
            for name in rm_states:
                check(torch.equal(got['rm'][name]['weight'], rm_states[name]['weight']))
            for name in pp_states:
                check(torch.equal(got['policy'][name],pp_states[name]))
    add("权重磁盘往返", save_reload_test)
    return tests


def run_checks() -> dict:
    tests = make_tests()
    for name, fn in tests:
        try:
            fn()
        except Exception as exc:
            raise AssertionError(f"固定检查失败：{name}") from exc
    return {"count": len(tests), "passed": [name for name, _ in tests],
            "scope": "标准库与CPU PyTorch，真实小张量训练，不是LLM成绩"}

### A：稳定的成对偏好损失

实现 `pairwise_loss_ex(chosen,rejected)`，返回标量均值。检查数值、梯度、极端分数和共同平移。

In [22]:
def pairwise_loss_ex(chosen,rejected):
    raise NotImplementedError("由你填写稳定的偏好排序损失")

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    exercise_check_a(pairwise_loss_ex)
    print("练习A通过")

### B：回答序列log概率

实现 `completion_logps_ex(logits,input_ids,completion_mask)`，返回每条回答的log概率和及目标数。只移位一次，忽略提示与padding，不误删有效结束符，不改成token均值。

In [23]:
def completion_logps_ex(logits,input_ids,completion_mask):
    raise NotImplementedError("由你填写shift、log_softmax、gather、mask和sum")

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    exercise_check_b(completion_logps_ex)
    print("练习B通过")

### C：固定参考的DPO损失

实现 `dpo_loss_ex(pc,pr,rc,rr,beta)`，四个输入依次是policy与reference对chosen/rejected的序列log概率。不能省略参考差，reference不收梯度。

policy与reference相同时loss为log(2)，但不意味着policy梯度为零。

In [24]:
def dpo_loss_ex(pc,pr,rc,rr,beta=.5):
    raise NotImplementedError("由你填写DPO，并冻结参考梯度")

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    exercise_check_c(dpo_loss_ex)
    print("练习C通过")

## 11. 参考实现与故障实验

先独立完成，再展开本单元。参考函数使用独立名称，不覆盖你的实现。

六种错误实现会真的运行：偏好方向反转、漏掉reference、参考参与反向、序列误求均值、没移位、截断当负例。只有检查确实失败，才记录“发现故障”。

In [25]:
def pairwise_loss_reference(chosen, rejected):
    aligned_vectors(chosen, rejected)
    return -F.logsigmoid(chosen - rejected).mean()


def completion_logps_reference(logits, input_ids, completion_mask):
    # 练习参考以本课已验证的输入前提写成另一条实现路径。
    logp = F.log_softmax(logits, dim=-1)
    sums, counts = [], []
    for b in range(input_ids.shape[0]):
        values = []
        for pos in range(1, input_ids.shape[1]):
            if bool(completion_mask[b,pos]):
                values.append(logp[b,pos-1,input_ids[b,pos]])
        require(bool(values), "不能没有回答目标")
        sums.append(torch.stack(values).sum())
        counts.append(len(values))
    return torch.stack(sums), torch.tensor(counts, device=input_ids.device)


def dpo_loss_reference(pc, pr, rc, rr, beta=0.5):
    aligned_vectors(pc, pr, rc, rr)
    advantage = (pc - rc.detach()) - (pr - rr.detach())
    return -F.logsigmoid(beta * advantage).mean()


def check_references() -> dict:
    exercise_check_a(pairwise_loss_reference)
    exercise_check_b(completion_logps_reference)
    exercise_check_c(dpo_loss_reference)
    return {"A": "passed", "B": "passed", "C": "passed"}


def run_mutations() -> dict:
    """真正替换实现并执行检查；不是直接写一份声称错误已被发现的报告。"""
    mutations = []
    def caught(name, fn):
        try:
            fn()
        except (AssertionError, LessonError):
            mutations.append({"name": name, "caught": True})
            return
        raise AssertionError(f"故障未被发现：{name}")
    caught("奖励偏好方向反转", lambda: exercise_check_a(lambda c,r: F.softplus(c-r).mean()))
    caught("DPO遗漏reference", lambda: exercise_check_c(lambda pc,pr,rc,rr,beta: F.softplus(-beta*(pc-pr)).mean()))
    caught("reference参与反向", lambda: exercise_check_c(lambda pc,pr,rc,rr,beta: F.softplus(-beta*(pc-pr-rc+rr)).mean()))
    def wrong_mean(logits, ids, mask):
        sums, counts = completion_logps(logits, ids, mask)
        return sums/counts, counts
    caught("回答log概率误用均值", lambda: exercise_check_b(wrong_mean))
    def wrong_shift(logits, ids, mask):
        logp = F.log_softmax(logits, -1).gather(-1, ids.unsqueeze(-1)).squeeze(-1)
        return (logp*mask).sum(-1), mask.sum(-1)
    caught("next-token未移位", lambda: exercise_check_b(wrong_shift))
    original = globals()['grade_candidate']
    def wrong_incomplete(task, rec):
        grade = original(task, rec)
        if grade['state'] == 'incomplete':
            return {'state':'graded','score':0.,'eligible':True}
        return grade
    try:
        globals()['grade_candidate'] = wrong_incomplete
        tasks, records, _ = make_fixture()
        bad = next(r for r in records if r['stop_reason']=='length')
        caught("把截断输出当负例", lambda: check(not grade_candidate(tasks[0], bad)['eligible']))
    finally:
        globals()['grade_candidate'] = original
    return {"count":len(mutations),"mutations":mutations}

In [26]:
print("参考实现:",check_references())
print(json.dumps(run_mutations(),ensure_ascii=False,indent=2))

参考实现: {'A': 'passed', 'B': 'passed', 'C': 'passed'}
{
  "count": 6,
  "mutations": [
    {
      "name": "奖励偏好方向反转",
      "caught": true
    },
    {
      "name": "DPO遗漏reference",
      "caught": true
    },
    {
      "name": "reference参与反向",
      "caught": true
    },
    {
      "name": "回答log概率误用均值",
      "caught": true
    },
    {
      "name": "next-token未移位",
      "caught": true
    },
    {
      "name": "把截断输出当负例",
      "caught": true
    }
  ]
}


## 12. 完整运行与重载

下方重新执行固定配置并保存全部报告，使用当前工作目录下的新时间戳子目录，不覆盖已有实验，也不联网。

奖励模型的固定留出结果不用于选择轮次。三个有限策略没有独立泛化评测。

In [27]:
def save_experiment(root: str | Path = 'lesson_57_outputs', data_only: bool = False) -> Path:
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    folder = root / ('run_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    folder.mkdir(exist_ok=False)
    manifest = {'version':VERSION, 'python':platform.python_version(), 'torch':torch.__version__,
                'device':'cpu', 'dtype':'float64', 'data_only':data_only,
                'status':'running', 'sources':SOURCES,
                'boundaries':['无模型下载','无真实LLM训练','合成偏好与人工特征','非PPO/GRPO复现']}
    write_json(folder/'manifest.json', manifest)
    try:
        tasks, records, contract = make_fixture()
        data = build_preferences(tasks, records)
        write_json(folder/'tasks_grader_only.json',tasks)
        write_jsonl(folder/'candidate_records.jsonl',records)
        write_json(folder/'data_contract.json',contract)
        write_json(folder/'data_audit.json',data)
        for split in ['train','dev','test']:
            write_jsonl(folder/f'pairs_{split}.jsonl',[p for p in data['pairs'] if p['split']==split])
        write_jsonl(folder/'generation_requests.jsonl',[
            {key:task[key] for key in ['task_id','group_id','split','prompt','planned_ids']}
            for task in tasks])
        write_json(folder/'mechanisms.json',objective_mechanisms())
        if not data_only:
            checks = run_checks()
            write_json(folder/'checks.json',checks)
            write_json(folder/'exercises.json',check_references())
            write_json(folder/'mutations.json',run_mutations())
            rm, rm_states = train_reward_experiment()
            policy, policy_states = train_policy_experiment()
            write_json(folder/'reward_models.json',rm)
            write_json(folder/'policy_objectives.json',policy)
            torch.save({'version':VERSION,'reward_models':rm_states,'policies':policy_states},folder/'small_models.pt')
            write_json(folder/'reload.json',reload_run(folder))
        manifest['status'] = 'completed'
        write_json(folder/'manifest.json',manifest)
        return folder
    except Exception as exc:
        manifest['status']='failed'; manifest['error']=type(exc).__name__+': '+str(exc)
        write_json(folder/'manifest.json',manifest)
        raise


def read_bounded_json(path: Path):
    require(path.stat().st_size <= 8*1024*1024, "报告过大")
    return json.loads(path.read_text(encoding='utf-8'))


def reload_run(folder: str | Path) -> dict:
    folder = Path(folder)
    payload = torch.load(folder/'small_models.pt', map_location='cpu', weights_only=True)
    require(payload.get('version') == VERSION, "检查点版本不符")
    rm_report = read_bounded_json(folder/'reward_models.json')
    pol_report = read_bounded_json(folder/'policy_objectives.json')
    differences = {}
    for name, state in payload['reward_models'].items():
        model=nn.Linear(2,1,bias=False).to(dtype=DTYPE)
        model.load_state_dict(state,strict=True)
        close(model.weight.flatten(),rm_report[name]['weights'])
        result=evaluate_ranker(model,'shifted')
        close(result['loss'],rm_report[name]['shifted_test']['loss'])
        differences['reward_'+name]=abs(result['loss']-rm_report[name]['shifted_test']['loss'])
    for name, tensor in payload['policies'].items():
        if name=='reference_logps':
            close(tensor.exp(),pol_report['contract']['initial_probabilities'])
        else:
            p=tensor.softmax(0)
            target=torch.tensor(pol_report[name]['probabilities'],dtype=DTYPE)
            close(p,target)
            differences['policy_'+name]=float((p-target).abs().max())
    tasks = read_bounded_json(folder/'tasks_grader_only.json')
    records = [json.loads(line) for line in (folder/'candidate_records.jsonl').read_text(encoding='utf-8').splitlines()]
    data = build_preferences(tasks,records)
    check(data==read_bounded_json(folder/'data_audit.json'),"重建偏好审计不一致")
    contract = read_bounded_json(folder/'data_contract.json')
    check(contract['task_digest'] == digest(tasks) and contract['record_digest'] == digest(records),
          "数据身份与契约不一致")
    for split in ['train','dev','test']:
        exported = [json.loads(line) for line in (folder/f'pairs_{split}.jsonl').read_text(encoding='utf-8').splitlines()]
        check(exported == [p for p in data['pairs'] if p['split'] == split], "导出偏好与重建不一致")
    return {'status':'passed','max_absolute_difference':max(differences.values()),
            'differences':differences,'preference_data_rebuilt':True,
            'source_files_modified':False,
            'boundary':'仅加载可信本课文件；weights_only不是完整安全认证'}


def print_summary(folder: Path) -> None:
    data=read_bounded_json(folder/'data_audit.json')
    print('偏好数据：',canonical(data['summary']))
    if (folder/'checks.json').exists():
        print(f"固定检查通过：{read_bounded_json(folder/'checks.json')['count']}项。")
        rm=read_bounded_json(folder/'reward_models.json')
        for kind in ['confounded','balanced']:
            print(kind, '训练排序=',rm[kind]['train']['pair_accuracy'],
                  '反相关留出排序=',rm[kind]['shifted_test']['pair_accuracy'])
        pp=read_bounded_json(folder/'policy_objectives.json')
        for kind in ['sft','dpo','exact_reward_kl']:
            print(kind,'最终三动作概率=',[round(x,6) for x in pp[kind]['probabilities']])
    print('输出目录：',folder.resolve())
    print('真实CPU小张量实验；不是语言模型能力、GPU性能或完整RLHF成绩。')


def main(argv=None) -> None:
    parser=argparse.ArgumentParser(description=__doc__)
    mode=parser.add_mutually_exclusive_group()
    mode.add_argument('--checks-only',action='store_true')
    mode.add_argument('--data-only',action='store_true')
    mode.add_argument('--math-only',action='store_true')
    mode.add_argument('--reload-run',type=Path)
    parser.add_argument('--output-dir',default='lesson_57_outputs')
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    torch.manual_seed(57)
    if args.checks_only:
        checks=run_checks()
        print(f"固定检查通过：{checks['count']}项。CPU数学与真实小张量训练，不是LLM成绩。")
    elif args.math_only:
        print(json.dumps(objective_mechanisms(),ensure_ascii=False,indent=2))
    elif args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2))
    else:
        folder=save_experiment(args.output_dir,data_only=args.data_only)
        print_summary(folder)

In [28]:
NOTEBOOK_RUN=save_experiment(OUTPUT_ROOT)
print_summary(NOTEBOOK_RUN)
print("重载核对:")
print(json.dumps(reload_run(NOTEBOOK_RUN),ensure_ascii=False,indent=2))

偏好数据： {"all_strict_pairs_before_cap":96,"pairs":24,"pairs_by_split":{"dev":4,"test":4,"train":16},"planned_slots":192,"received_records":168,"states":{"generation_error":24,"graded":96,"incomplete":24,"invalid_format":24,"missing":24},"ties_skipped":48}
固定检查通过：74项。
confounded 训练排序= 1.0 反相关留出排序= 0.0
balanced 训练排序= 1.0 反相关留出排序= 1.0
sft 最终三动作概率= [0.973351, 0.016656, 0.009993]
dpo 最终三动作概率= [0.942541, 0.00344, 0.054019]
exact_reward_kl 最终三动作概率= [0.837882, 0.074765, 0.087353]
输出目录： /mnt/data/_lesson57_work/notebook_run/lesson_57_outputs/run_20261102T150949_871363Z
真实CPU小张量实验；不是语言模型能力、GPU性能或完整RLHF成绩。
重载核对:
{
  "status": "passed",
  "max_absolute_difference": 0.0,
  "differences": {
    "reward_confounded": 0.0,
    "reward_balanced": 0.0,
    "policy_sft": 0.0,
    "policy_dpo": 0.0,
    "policy_exact_reward_kl": 0.0
  },
  "preference_data_rebuilt": true,
  "source_files_modified": false,
  "boundary": "仅加载可信本课文件；weights_only不是完整安全认证"
}


### 文件与终端命令

| 文件 | 用途 |
|---|---|
| `generation_requests.jsonl` | 提示与计划，不含答案 |
| `tasks_grader_only.json` | 评分侧参考值 |
| `candidate_records.jsonl` | 手工响应、失败与截断 |
| `pairs_train/dev/test.jsonl` | 已固定划分的偏好与来源 |
| `data_audit.json` | 候选资格、平分、缺失、配对上限 |
| `reward_models.json` | 奖励模型训练与留出结果 |
| `policy_objectives.json` | 三种策略目标的记录 |
| `mechanisms.json` | 数学、梯度、长度与代理反例 |
| `small_models.pt` | 小评分器与策略张量 |
| `reload.json` | 磁盘重载一致性 |

```bash
python lesson_57_rewards_preferences_and_posttraining.py --data-only
python lesson_57_rewards_preferences_and_posttraining.py --math-only
python lesson_57_rewards_preferences_and_posttraining.py --reload-run lesson_57_outputs/实际目录
```

`--data-only`不训练，只生成数据和数学报告；导入脚本仍需PyTorch。重载仅用于恢复模型与核对数据，不包含精确续训的完整优化器、随机状态和进度。仅加载可信本课文件，`weights_only=True`不是完整安全认证。

## 13. 验收问题

**数据：** 为什么运行报错不直接当rejected？为什么等价正确答案不强行排序？为什么先按来源切分？

**梯度：** 评分模型训练后，生成参数是否改变？为什么只对一个独立奖励数字反传，不能训练生成器？

**目标：** 为什么DPO能改善相对比值却降低chosen绝对概率？为什么求token均值会改变原始序列目标？

**评测：** 两个评分模型训练全对，为什么在固定反向相关条件下一胜一败？三动作概率表为什么不是语言能力结果？

下一课是 **第58课：从零实现语言模型DPO——参考模型、成对批次与LoRA更新**，将这里的公式接回自回归模型。

## 原始资料与边界

| 资料 | 目的 |
|---|---|
| [InstructGPT](https://arxiv.org/abs/2203.02155) | 区分SFT、RM与策略优化 |
| [TRL Reward Modeling](https://huggingface.co/docs/trl/reward_trainer) | 偏好格式与成对损失 |
| [DPO论文](https://arxiv.org/abs/2305.18290) | 固定参考与直接偏好目标 |
| [DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer) | 数据与损失变体的约定 |
| [Reward Overoptimization](https://arxiv.org/abs/2210.10760) | 代理奖励与真实目标分离 |
| [DeepSeekMath](https://arxiv.org/abs/2402.03300) | GRPO的组成与背景 |
| [PyTorch log_softmax](https://docs.pytorch.org/docs/2.10/generated/torch.nn.functional.log_softmax.html) | 稳定log概率计算 |

讲义保留实际CPU输出。没有真实LLM、TRL、PEFT、CUDA或显存性能测试，不是完整RLHF/GRPO复现，也不是自然语言奖励模型benchmark。